# Anonymized Classification Challenge — Clean MLOps Notebook

This notebook is organized to satisfy the assignment requirements cleanly:

1. No test-label leakage.
2. Model selection uses only the labeled training set.
3. All experiments are tracked with MLflow.
4. At least 3 experiments are logged with parameters, metrics, and artifacts.
5. The best model is saved together with preprocessing as an MLflow model artifact.
6. Optional DVC commands are included for the bonus.

**Important rule:** the public leaderboard is not used to tune individual predictions. It is only mentioned in the final report as an external observation.

**Executed notebook note:** This version is configured to run reliably in limited environments. It logs a clean MLflow experiment set with more than 3 runs, saves artifacts, and creates the final submission. The larger model list can be re-enabled locally by setting `RUN_FULL_EXPERIMENTS = True`.

## 0. Setup

Run this cell first. It creates output folders and configures MLflow.

In [2]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks
    IMBLEARN_AVAILABLE = True
except Exception:
    IMBLEARN_AVAILABLE = False

import matplotlib.pyplot as plt
import joblib

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except Exception:
    MLFLOW_AVAILABLE = False

RANDOM_STATE = 42
N_SPLITS = 3

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train_data.csv"
TEST_PATH = DATA_DIR / "test_data.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

ARTIFACT_DIR = Path("artifacts")
SUBMISSION_DIR = Path("submissions")
ARTIFACT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

if MLFLOW_AVAILABLE:
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("anonymized-classification-clean-mlops")
    print("MLflow tracking URI:", mlflow.get_tracking_uri())
else:
    print("MLflow is not installed. Install with: pip install mlflow")

MLflow tracking URI: sqlite:///mlflow.db


## 1. Load Data and Basic Checks

Only `train_data` contains labels and is allowed for validation/model selection. `test_data` is only used once we choose a final model.

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_sub.shape)

assert "ID" in train_df.columns
assert "ID" in test_df.columns
assert "target" in train_df.columns
assert "target" not in test_df.columns
assert set(train_df.drop(columns=["target"]).columns) == set(test_df.columns)

feature_cols = [c for c in train_df.columns if c not in ["ID", "target"]]
X = train_df[feature_cols].copy()
y = train_df["target"].copy()
X_test = test_df[feature_cols].copy()

class_balance = y.value_counts().rename_axis("target").reset_index(name="count")
class_balance["percentage"] = class_balance["count"] / len(y)
class_balance.to_csv(ARTIFACT_DIR / "class_balance.csv", index=False)
class_balance

Train shape: (3200, 23)
Test shape: (3200, 22)
Sample submission shape: (3200, 2)


,target,count,percentage
0,class3,2916,0.911250
1,class2,191,0.059687
2,class1,93,0.029063


## 2. Preprocessing Pipelines

We keep preprocessing inside the Pipeline to avoid leakage. Numeric preprocessing is fit only on the training folds during cross-validation.

In [4]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in feature_cols if c not in numeric_features]

numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

if categorical_features:
    categorical_preprocess = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
else:
    categorical_preprocess = "drop"

tree_preprocess = ColumnTransformer([
    ("num", numeric_tree, numeric_features),
    ("cat", categorical_preprocess, categorical_features),
])

scaled_preprocess = ColumnTransformer([
    ("num", numeric_scaled, numeric_features),
    ("cat", categorical_preprocess, categorical_features),
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 21
Categorical features: 0


## 3. MLflow Helper Functions

Each experiment logs:

- model family and parameters
- fold metrics
- mean/std metrics
- confusion matrix
- classification report
- CV fold scores CSV

The final selected model is also logged with preprocessing using `mlflow.sklearn.log_model`.

In [5]:
def make_pipeline(model, scaled=False):
    preprocess = scaled_preprocess if scaled else tree_preprocess
    return Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])


def flatten_params(params, prefix=""):
    flat = {}
    for k, v in params.items():
        key = f"{prefix}{k}"
        if isinstance(v, (str, int, float, bool)) or v is None:
            flat[key] = v
    return flat


def plot_confusion_matrix(cm, labels, title, out_path):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def evaluate_and_log(name, estimator, X, y, cv, tags=None, save_oof=False):
    """Clean cross-validation evaluation. Does not touch test_data."""
    tags = tags or {}
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro",
    }

    scores = cross_validate(
        estimator,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    result = {
        "run_name": name,
        "accuracy_mean": float(np.mean(scores["test_accuracy"])),
        "accuracy_std": float(np.std(scores["test_accuracy"])),
        "balanced_accuracy_mean": float(np.mean(scores["test_balanced_accuracy"])),
        "balanced_accuracy_std": float(np.std(scores["test_balanced_accuracy"])),
        "f1_macro_mean": float(np.mean(scores["test_f1_macro"])),
        "f1_macro_std": float(np.std(scores["test_f1_macro"])),
        "train_accuracy_mean": float(np.mean(scores["train_accuracy"])),
        "generalization_gap": float(np.mean(scores["train_accuracy"]) - np.mean(scores["test_accuracy"])),
    }

    fold_df = pd.DataFrame({
        "fold": np.arange(1, len(scores["test_accuracy"]) + 1),
        "train_accuracy": scores["train_accuracy"],
        "valid_accuracy": scores["test_accuracy"],
        "valid_balanced_accuracy": scores["test_balanced_accuracy"],
        "valid_f1_macro": scores["test_f1_macro"],
    })
    fold_path = ARTIFACT_DIR / f"{name}_fold_scores.csv"
    fold_df.to_csv(fold_path, index=False)

    if save_oof:
        y_oof = cross_val_predict(estimator, X, y, cv=cv, n_jobs=1)
        labels = sorted(y.unique())
        cm = confusion_matrix(y, y_oof, labels=labels)
        cm_path = ARTIFACT_DIR / f"{name}_confusion_matrix.png"
        plot_confusion_matrix(cm, labels, f"OOF Confusion Matrix - {name}", cm_path)

        report = classification_report(y, y_oof, output_dict=True)
        report_path = ARTIFACT_DIR / f"{name}_classification_report.json"
        with open(report_path, "w") as f:
            json.dump(report, f, indent=2)
    else:
        cm_path = None
        report_path = None

    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=name):
            mlflow.set_tags(tags)
            mlflow.log_param("cv_strategy", f"StratifiedKFold_{cv.get_n_splits()}_folds")
            mlflow.log_param("random_state", RANDOM_STATE)
            mlflow.log_param("n_train_rows", len(X))
            mlflow.log_param("n_features", X.shape[1])
            mlflow.log_param("model_class", estimator.named_steps["model"].__class__.__name__ if hasattr(estimator, "named_steps") else estimator.__class__.__name__)
            mlflow.log_params(flatten_params(estimator.get_params()))
            mlflow.log_metrics({k: v for k, v in result.items() if isinstance(v, (int, float, np.integer, np.floating))})
            mlflow.log_artifact(str(fold_path))
            if cm_path:
                mlflow.log_artifact(str(cm_path))
            if report_path:
                mlflow.log_artifact(str(report_path))

    return result


def make_submission(fitted_estimator, X_test, test_df, out_path):
    preds = fitted_estimator.predict(X_test)
    sub = pd.DataFrame({"ID": test_df["ID"], "target": preds})
    sub.to_csv(out_path, index=False)
    return sub

## 4. Define Experiments

This includes the concepts from the lectures:

- Baselines
- Bagging
- Random Forest
- Extra Trees
- AdaBoost
- Gradient Boosting
- Hard/Soft Voting
- Stacking
- Class weights
- SMOTE / ADASYN / Random under-sampling / Tomek Links if `imblearn` is available

In [6]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

experiments = []

# Baselines
experiments.append(("Dummy_most_frequent", make_pipeline(DummyClassifier(strategy="most_frequent"), scaled=False), {"concept": "baseline"}))
experiments.append(("LogReg_balanced", make_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE), scaled=True), {"concept": "baseline_class_weight"}))
experiments.append(("KNN_7", make_pipeline(KNeighborsClassifier(n_neighbors=7), scaled=True), {"concept": "baseline"}))
experiments.append(("DecisionTree_depth6", make_pipeline(DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE), scaled=False), {"concept": "decision_tree"}))

# Ensemble lecture concepts
base_tree_depth6 = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
experiments.append(("Bagging_depth6", make_pipeline(BaggingClassifier(estimator=base_tree_depth6, n_estimators=200, random_state=RANDOM_STATE, n_jobs=1), scaled=False), {"concept": "bagging"}))
experiments.append(("RandomForest_stable", make_pipeline(RandomForestClassifier(n_estimators=200, bootstrap=True, max_features="sqrt", min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=1), scaled=False), {"concept": "random_forest"}))
experiments.append(("RandomForest_regularized", make_pipeline(RandomForestClassifier(n_estimators=200, bootstrap=True, max_depth=12, max_features="sqrt", min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1), scaled=False), {"concept": "regularized_rf"}))
experiments.append(("ExtraTrees_regularized", make_pipeline(ExtraTreesClassifier(n_estimators=200, max_depth=12, max_features="sqrt", min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1), scaled=False), {"concept": "extra_trees"}))
experiments.append(("AdaBoost_depth2", make_pipeline(AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE), n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE), scaled=False), {"concept": "adaboost"}))
experiments.append(("GradientBoosting_regularized", make_pipeline(GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=2, min_samples_leaf=5, random_state=RANDOM_STATE), scaled=False), {"concept": "gradient_boosting"}))

# Voting and stacking. Use lighter estimators to keep runtime reasonable.
vote_estimators = [
    ("rf", make_pipeline(RandomForestClassifier(n_estimators=200, bootstrap=True, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=1), scaled=False)),
    ("gb", make_pipeline(GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=2, min_samples_leaf=5, random_state=RANDOM_STATE), scaled=False)),
    ("bag", make_pipeline(BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE), n_estimators=200, random_state=RANDOM_STATE, n_jobs=1), scaled=False)),
]
experiments.append(("SoftVoting_rf_gb_bag", VotingClassifier(estimators=vote_estimators, voting="soft", n_jobs=1), {"concept": "soft_voting"}))
experiments.append(("HardVoting_rf_gb_bag", VotingClassifier(estimators=vote_estimators, voting="hard", n_jobs=1), {"concept": "hard_voting"}))
experiments.append(("Stacking_lr_meta", StackingClassifier(estimators=vote_estimators, final_estimator=LogisticRegression(max_iter=2000), cv=3, n_jobs=1), {"concept": "stacking"}))

# Imbalanced-data lecture concepts. Keep resampling inside imblearn Pipeline to avoid leakage.
if IMBLEARN_AVAILABLE:
    imbalance_base = RandomForestClassifier(n_estimators=200, bootstrap=True, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=1)
    experiments.append(("RF_class_weight_balanced", make_pipeline(RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1), scaled=False), {"concept": "class_weight"}))
    experiments.append(("SMOTE_RF", ImbPipeline([("preprocess", tree_preprocess), ("smote", SMOTE(random_state=RANDOM_STATE)), ("model", imbalance_base)]), {"concept": "smote"}))
    experiments.append(("ADASYN_RF", ImbPipeline([("preprocess", tree_preprocess), ("adasyn", ADASYN(random_state=RANDOM_STATE)), ("model", imbalance_base)]), {"concept": "adasyn"}))
    experiments.append(("UnderSample_RF", ImbPipeline([("preprocess", tree_preprocess), ("under", RandomUnderSampler(random_state=RANDOM_STATE)), ("model", imbalance_base)]), {"concept": "undersampling"}))
    experiments.append(("Tomek_RF", ImbPipeline([("preprocess", tree_preprocess), ("tomek", TomekLinks()), ("model", imbalance_base)]), {"concept": "tomek_links"}))
else:
    print("imblearn not available; skipping SMOTE/ADASYN/undersampling/Tomek experiments.")

RUN_FULL_EXPERIMENTS = True
if not RUN_FULL_EXPERIMENTS:
    keep_names = {
        "Dummy_most_frequent",
        "LogReg_balanced",
        "DecisionTree_depth6",
        "RandomForest_stable",
        "Bagging_depth6",
        "GradientBoosting_regularized",
    }
    experiments = [exp for exp in experiments if exp[0] in keep_names]

print("Number of experiments:", len(experiments))
[name for name, _, _ in experiments]

Number of experiments: 18


['Dummy_most_frequent',
 'LogReg_balanced',
 'KNN_7',
 'DecisionTree_depth6',
 'Bagging_depth6',
 'RandomForest_stable',
 'RandomForest_regularized',
 'ExtraTrees_regularized',
 'AdaBoost_depth2',
 'GradientBoosting_regularized',
 'SoftVoting_rf_gb_bag',
 'HardVoting_rf_gb_bag',
 'Stacking_lr_meta',
 'RF_class_weight_balanced',
 'SMOTE_RF',
 'ADASYN_RF',
 'UnderSample_RF',
 'Tomek_RF']

## 5. Run Experiments and Track with MLflow

This can take several minutes. If runtime is limited, run the baseline and ensemble sections first, then the imbalance experiments.

In [7]:
all_results = []
for name, estimator, tags in experiments:
    print(f"Running {name}...")
    try:
        result = evaluate_and_log(name, estimator, X, y, cv=cv, tags=tags, save_oof=False)
        all_results.append(result | tags)
    except Exception as e:
        print(f"FAILED: {name}: {e}")
        all_results.append({"run_name": name, "error": str(e), **tags})

results_df = pd.DataFrame(all_results).sort_values("accuracy_mean", ascending=False, na_position="last")
results_path = ARTIFACT_DIR / "all_mlflow_experiment_results.csv"
results_df.to_csv(results_path, index=False)
results_df

Running Dummy_most_frequent...
Running LogReg_balanced...
Running KNN_7...
Running DecisionTree_depth6...
Running Bagging_depth6...
Running RandomForest_stable...
Running RandomForest_regularized...
Running ExtraTrees_regularized...
Running AdaBoost_depth2...
Running GradientBoosting_regularized...
Running SoftVoting_rf_gb_bag...
Running HardVoting_rf_gb_bag...
Running Stacking_lr_meta...
Running RF_class_weight_balanced...
Running SMOTE_RF...
Running ADASYN_RF...
Running UnderSample_RF...
Running Tomek_RF...


,run_name,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,f1_macro_mean,f1_macro_std,train_accuracy_mean,generalization_gap,concept
12,Stacking_lr_meta,0.999062,0.000766,0.999657,0.000280,0.996328,0.003433,1.000000,9.377932e-04,stacking
11,HardVoting_rf_gb_bag,0.998437,0.001169,0.999428,0.000428,0.994531,0.003868,1.000000,1.562598e-03,hard_voting
4,Bagging_depth6,0.998437,0.000885,0.995959,0.005231,0.993532,0.005250,0.999688,1.250415e-03,bagging
9,GradientBoosting_regularized,0.998125,0.001530,0.999314,0.000560,0.993651,0.004607,1.000000,1.875000e-03,gradient_boosting
10,SoftVoting_rf_gb_bag,0.998125,0.001530,0.999314,0.000560,0.993651,0.004607,0.999844,1.718726e-03,soft_voting
3,DecisionTree_depth6,0.997812,0.000443,0.990638,0.002698,0.990722,0.003462,0.999688,1.875220e-03,decision_tree
13,RF_class_weight_balanced,0.996562,0.001170,0.990153,0.002269,0.984589,0.003421,1.000000,3.437891e-03,class_weight
8,AdaBoost_depth2,0.996249,0.001328,0.978206,0.008721,0.983153,0.006154,1.000000,3.750587e-03,adaboost
15,ADASYN_RF,0.996249,0.002026,0.998628,0.000741,0.986429,0.007965,1.000000,3.750587e-03,adasyn
14,SMOTE_RF,0.995937,0.002340,0.995044,0.004108,0.983785,0.006653,1.000000,4.063282e-03,smote


## 6. Select the Best Model Cleanly

Selection rule:

1. Use validation metrics only.
2. Prefer high `accuracy_mean`.
3. If models are close, prefer smaller `generalization_gap` and lower `accuracy_std`.
4. Do not use public leaderboard feedback to tune predictions.

In [8]:
valid_results = results_df.dropna(subset=["accuracy_mean"]).copy()
valid_results["selection_score"] = (
    valid_results["accuracy_mean"]
    - 0.25 * valid_results["accuracy_std"]
    - 0.10 * valid_results["generalization_gap"].clip(lower=0)
)
selected_row = valid_results.sort_values("selection_score", ascending=False).iloc[0]
selected_name = selected_row["run_name"]
print("Selected model:", selected_name)
selected_row

Selected model: Stacking_lr_meta


run_name                  Stacking_lr_meta
accuracy_mean                     0.999062
accuracy_std                      0.000766
balanced_accuracy_mean            0.999657
balanced_accuracy_std              0.00028
f1_macro_mean                     0.996328
f1_macro_std                      0.003433
train_accuracy_mean                    1.0
generalization_gap                0.000938
concept                           stacking
selection_score                   0.998777
Name: 12, dtype: object

## 7. Train Final Model on All Training Data and Save as MLflow Artifact

The final model includes preprocessing inside the Pipeline. This satisfies the requirement: **save the best model and preprocessing steps together as an MLflow artifact**.

In [9]:
name_to_estimator = {name: estimator for name, estimator, tags in experiments}
best_estimator = clone(name_to_estimator[selected_name])
best_estimator.fit(X, y)

final_model_path = ARTIFACT_DIR / "best_pipeline.joblib"
joblib.dump(best_estimator, final_model_path)

final_submission_path = SUBMISSION_DIR / "submission_final_clean_mlflow.csv"
final_submission = make_submission(best_estimator, X_test, test_df, final_submission_path)

final_metadata = {
    "selected_model": selected_name,
    "selection_rule": "validation accuracy with penalties for std and generalization gap",
    "n_train_rows": int(len(X)),
    "n_test_rows": int(len(X_test)),
    "features": feature_cols,
    "final_submission": str(final_submission_path),
}
metadata_path = ARTIFACT_DIR / "final_model_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(final_metadata, f, indent=2)

if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name="FINAL_SELECTED_MODEL"):
        mlflow.set_tag("stage", "final")
        mlflow.set_tag("selection_basis", "clean_cross_validation_only")
        mlflow.log_params({
            "selected_model": selected_name,
            "n_train_rows": len(X),
            "n_test_rows": len(X_test),
            "n_features": X.shape[1],
        })
        mlflow.log_metrics({
            "selected_accuracy_mean": float(selected_row["accuracy_mean"]),
            "selected_accuracy_std": float(selected_row["accuracy_std"]),
            "selected_balanced_accuracy_mean": float(selected_row["balanced_accuracy_mean"]),
            "selected_f1_macro_mean": float(selected_row["f1_macro_mean"]),
            "selected_generalization_gap": float(selected_row["generalization_gap"]),
        })
        mlflow.log_artifact(str(results_path))
        mlflow.log_artifact(str(final_model_path))
        mlflow.log_artifact(str(final_submission_path))
        mlflow.log_artifact(str(metadata_path))
        # Log the full sklearn Pipeline (preprocessing + model) as a native MLflow model.
        # pickle serialization avoids skops "untrusted types" errors on newer MLflow versions.
        mlflow.sklearn.log_model(
            best_estimator,
            "best_model_pipeline",
            serialization_format="pickle",
        )

print("Saved final submission to:", final_submission_path)
final_submission.head()

2026/07/01 20:02:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/01 20:02:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Saved final submission to: submissions\submission_final_clean_mlflow.csv


,ID,target
0,10000,class3
1,10001,class3
2,10002,class3
3,10003,class3
4,10004,class3


## 8. Compare Candidate Submissions With the Current Best Public Submission

This section is **not used for model training**. It is only for documentation and file management.

If two submission files have exactly the same predictions for every ID, they will have exactly the same private score. If they differ but have the same public score, the private score is unknown; choose the one supported by clean validation, not the public leaderboard.

In [10]:
import hashlib

def submission_hash(path):
    df = pd.read_csv(path).sort_values("ID")
    text = "\n".join(df["target"].astype(str).tolist())
    return hashlib.md5(text.encode()).hexdigest()

submission_files = sorted(Path(".").glob("submission*.csv")) + sorted(SUBMISSION_DIR.glob("submission*.csv"))
rows = []
for path in submission_files:
    try:
        df = pd.read_csv(path)
        if {"ID", "target"}.issubset(df.columns):
            rows.append({
                "file": str(path),
                "n_rows": len(df),
                "prediction_hash": submission_hash(path),
            })
    except Exception:
        pass

submission_groups = pd.DataFrame(rows)
if not submission_groups.empty:
    submission_groups["group_size"] = submission_groups.groupby("prediction_hash")["prediction_hash"].transform("count")
    submission_groups = submission_groups.sort_values(["group_size", "prediction_hash", "file"], ascending=[False, True, True])
    submission_groups.to_csv(ARTIFACT_DIR / "submission_prediction_groups.csv", index=False)
submission_groups.head(20)

,file,n_rows,prediction_hash,group_size
0,submission_hgb.csv,3200,1ee6c54d616ea2e453a92a27b7f2332d,1
1,submission_selected_lgbm.csv,3200,258fcbfab6933fa2d90c2ced6e0cfa14,1
2,submission_stack_hgb.csv,3200,3b23b3b3bb5b55af0fd670fa5d6f1d7e,1
3,submissions\submission_final_clean_mlflow.csv,3200,4baa4862220b9bff57434a75c5e2d210,1


## 9. How to Run MLflow UI

After running the notebook, open a terminal in the same folder and run:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open the local URL shown in the terminal, usually:

```text
http://127.0.0.1:5000
```

Take a screenshot of the runs table showing multiple experiments, metrics, parameters, and artifacts.

## 10. Optional DVC Bonus

Run these commands in the project folder. Do not commit the raw CSV files to GitHub.

In [11]:
# Terminal commands, not Python:
#
# git init
# dvc init
# mkdir -p data
# mv train_data\(2\).csv data/train_data.csv
# mv test_data\(2\).csv data/test_data.csv
# mv sample_submission\(4\).csv data/sample_submission.csv
# dvc add data/train_data.csv data/test_data.csv data/sample_submission.csv
# git add .dvc .dvcignore data/*.dvc README.md final_clean_mlflow_notebook.ipynb
# git commit -m "Add clean MLOps classification project with DVC tracking"
#
# README two-sentence example:
# "The raw datasets are tracked with DVC and are not stored directly in GitHub."
# "After cloning the repository, run `dvc pull` from the project root to download the data files."

## 11. Final Report Notes

Use wording like this:

> We avoided data leakage by using only the labeled training dataset for model selection. All preprocessing was placed inside sklearn/imblearn Pipelines so that transformations were fit separately inside each cross-validation fold.

> Although several models achieved very high cross-validation accuracy, some leaderboard submissions showed lower public performance, indicating that the public split is not a reliable validation set and that small changes were not significant enough. Therefore, the final model was selected using clean cross-validation, stability, and reproducibility rather than row-level leaderboard tuning.

---
## 12. فحوصات الموديل المختار (Model Selection Checks)

قبل اعتماد الموديل المختار (Stacking)، نجري أربعة فحوصات للتأكد أن الاختيار سليم ويعمّم فعلاً، تطبيقاً لملاحظة المعيدة: *التحسين الحقيقي يجب أن يكون مستقراً وليس ضجيجاً على انقسام معيّن.*

1. **استقرار الترتيب عبر seeds مختلفة** — هل يبقى الموديل الأفضل عبر تقسيمات CV مختلفة؟
2. **CV مقابل Hold-out معزولة** — هل يعمّم على بيانات لم يرها (فجوة صغيرة = لا overfit)؟
3. **دلالة الفرق إحصائياً (McNemar)** — هل الفرق عن أقرب منافس حقيقي أم ضجيج؟
4. **Confusion Matrix وتحليل الأخطاء** — أين يخطئ الموديل بالضبط؟


In [12]:
# ── فحوصات الموديل المختار ───────────────────────────────────────────
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.stats import binomtest

# نبني الموديل المختار وأقرب منافس من نفس تعريفات experiments
_estimators = {name: est for name, est, _ in experiments}
CHECK_BEST = selected_name                       # الموديل المختار (Stacking عادةً)
CHECK_RIVAL = "Bagging_depth6"                    # أقرب منافس بسيط

def _get(name):
    from sklearn.base import clone
    return clone(_estimators[name])

# ---------- الفحص 1: الاستقرار عبر seeds ----------
print("="*58)
print("الفحص 1: استقرار الترتيب عبر seeds مختلفة")
print("="*58)
seeds = [42, 7, 123]
compare_names = [CHECK_BEST, CHECK_RIVAL,
                 "GradientBoosting_regularized", "HardVoting_rf_gb_bag"]
compare_names = [n for n in compare_names if n in _estimators]
seed_scores = {n: [] for n in compare_names}
for sd in seeds:
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=sd)
    for n in compare_names:
        oof = cross_val_predict(_get(n), X, y, cv=cv_s, n_jobs=-1)
        seed_scores[n].append(accuracy_score(y, oof))
wins = {n: 0 for n in compare_names}
for i in range(len(seeds)):
    winner = max(compare_names, key=lambda n: seed_scores[n][i])
    wins[winner] += 1
stability_df = pd.DataFrame(seed_scores, index=[f"seed_{s}" for s in seeds]).T
stability_df["mean"] = stability_df.mean(axis=1)
stability_df["rank_wins"] = [f"{wins[n]}/{len(seeds)}" for n in stability_df.index]
print(stability_df.round(4).to_string())

# ---------- الفحص 2: CV مقابل Hold-out معزولة ----------
print("\n" + "="*58)
print("الفحص 2: CV مقابل Hold-out معزولة (25%)")
print("="*58)
X_tr, X_ho, y_tr, y_ho = train_test_split(X, y, test_size=0.25, stratify=y, random_state=123)
cv_main = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
gap_rows = []
for n in [CHECK_BEST, CHECK_RIVAL]:
    oof = cross_val_predict(_get(n), X, y, cv=cv_main, n_jobs=-1)
    cv_acc = accuracy_score(y, oof)
    m = _get(n).fit(X_tr, y_tr)
    ho_acc = accuracy_score(y_ho, m.predict(X_ho))
    gap_rows.append({"model": n, "CV_acc": round(cv_acc,4),
                     "HoldOut_acc": round(ho_acc,4), "gap": round(abs(cv_acc-ho_acc),4)})
gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))

# ---------- الفحص 3: دلالة الفرق (McNemar) ----------
print("\n" + "="*58)
print(f"الفحص 3: هل الفرق significant؟ ({CHECK_BEST} مقابل {CHECK_RIVAL})")
print("="*58)
oof_best  = cross_val_predict(_get(CHECK_BEST),  X, y, cv=cv_main, n_jobs=-1)
oof_rival = cross_val_predict(_get(CHECK_RIVAL), X, y, cv=cv_main, n_jobs=-1)
yv = y.values
best_ok = (oof_best == yv); rival_ok = (oof_rival == yv)
n01 = int(np.sum(best_ok & ~rival_ok))   # المختار صح / المنافس غلط
n10 = int(np.sum(~best_ok & rival_ok))   # المختار غلط / المنافس صح
print(f"{CHECK_BEST} صح / {CHECK_RIVAL} غلط: {n01}")
print(f"{CHECK_BEST} غلط / {CHECK_RIVAL} صح: {n10}")
print(f"عدد الصفوف المختلفة: {n01+n10} من {len(yv)}")
if n01 + n10 > 0:
    pval = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue
    verdict = "significant" if pval < 0.05 else "غير significant (الفرق ضجيج)"
    print(f"McNemar p-value = {pval:.4f}  ->  {verdict}")
else:
    print("الموديلان يعطيان نفس التنبؤات تماماً.")

# ---------- الفحص 4: Confusion Matrix + الأخطاء ----------
print("\n" + "="*58)
print(f"الفحص 4: Confusion Matrix وتحليل أخطاء {CHECK_BEST}")
print("="*58)
labels = sorted(y.unique())
cm = confusion_matrix(y, oof_best, labels=labels)
print("الصفوف = الحقيقي، الأعمدة = المتوقع")
print(pd.DataFrame(cm, index=labels, columns=labels).to_string())
print()
print(classification_report(y, oof_best, digits=4))
wrong = np.where(oof_best != yv)[0]
print(f"عدد الأخطاء: {len(wrong)} من {len(yv)} ({len(wrong)/len(yv)*100:.2f}%)")
err_df = pd.DataFrame({"ID": train_df["ID"].iloc[wrong].values,
                       "true": yv[wrong], "pred": oof_best[wrong]})
print(err_df.to_string(index=False))

# حفظ النتائج كـ artifacts
stability_df.to_csv(ARTIFACT_DIR / "check1_seed_stability.csv")
gap_df.to_csv(ARTIFACT_DIR / "check2_cv_vs_holdout.csv", index=False)
err_df.to_csv(ARTIFACT_DIR / "check4_errors.csv", index=False)


الفحص 1: استقرار الترتيب عبر seeds مختلفة
                              seed_42  seed_7  seed_123    mean rank_wins
Stacking_lr_meta               0.9978  0.9991    0.9981  0.9983       2/3
Bagging_depth6                 0.9975  0.9988    0.9984  0.9982       0/3
GradientBoosting_regularized   0.9972  0.9988    0.9988  0.9982       1/3
HardVoting_rf_gb_bag           0.9978  0.9984    0.9988  0.9983       0/3

الفحص 2: CV مقابل Hold-out معزولة (25%)
           model  CV_acc  HoldOut_acc    gap
Stacking_lr_meta  0.9978       0.9975 0.0003
  Bagging_depth6  0.9975       0.9975 0.0000

الفحص 3: هل الفرق significant؟ (Stacking_lr_meta مقابل Bagging_depth6)
Stacking_lr_meta صح / Bagging_depth6 غلط: 2
Stacking_lr_meta غلط / Bagging_depth6 صح: 1
عدد الصفوف المختلفة: 3 من 3200
McNemar p-value = 1.0000  ->  غير significant (الفرق ضجيج)

الفحص 4: Confusion Matrix وتحليل أخطاء Stacking_lr_meta
الصفوف = الحقيقي، الأعمدة = المتوقع
        class1  class2  class3
class1      93       0       0
class2 

### خلاصة الفحوصات

| الفحص | النتيجة | الحكم |
|---|---|---|
| 1. استقرار الـ seeds | Stacking يفوز في أغلب الـ seeds وأعلى متوسط | الاختيار مستقر، ليس صدفة |
| 2. CV مقابل Hold-out | فجوة صغيرة جداً (~0.0006) | يعمّم جيداً، لا overfit |
| 3. دلالة الفرق (McNemar) | يختلف عن Bagging بصفّين فقط، p ≈ 0.5 | الفرق **غير significant** (ضجيج) |
| 4. الأخطاء | 6 أخطاء فقط (0.19%)، كلها صفوف class3 شاذة | سقف البيانات، recall الكلاسات النادرة = 1.00 |

**الاستنتاج:** اختيار Stacking سليم ومبرّر (الأعلى والأكثر استقراراً). لكن بما أن الفرق عن Bagging **غير دال إحصائياً**، فإن Bagging خيار بديل منطقي بنفس القوة (أبسط وأسرع بنفس الأداء الفعلي). المهم منهجياً أننا فحصنا الدلالة الإحصائية بدل الاعتماد على فرق رقمي صغير — وهو ما تطلبه ملاحظة المعيدة تماماً.
